# The first instant of thermal contact
Two long solids, initially at different uniform temperatures, touch at x = 0. Compare the interface temperature determined by thermal effusivity with whole temperature profiles, then check heat conservation. The experiment uses SI units and assumes perfect contact and short observation times.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

k_left, k_right=4.,1.            # W/(m K)
capacity_left,capacity_right=1e6,1e6  # volumetric heat capacities, J/(m³ K)
time=2.                           # seconds
TL,TR=80.,20.                     # degrees Celsius
assert min(k_left,k_right,capacity_left,capacity_right,time)>0
al,ar=k_left/capacity_left,k_right/capacity_right
el,er=np.sqrt(k_left*capacity_left),np.sqrt(k_right*capacity_right)
contact=(el*TL+er*TR)/(el+er)
erf=np.vectorize(math.erf,otypes=[float])
def temperature(x,t):
    return np.where(x<=0,contact+(TL-contact)*erf(-x/(2*np.sqrt(al*t))),
                    contact+(TR-contact)*erf(x/(2*np.sqrt(ar*t))))
def q(t):
    return el*er/(el+er)*(TL-TR)/np.sqrt(np.pi*t)
Q=2*q(time)*time
print(f"Interface {contact:.3f} °C; flux at t = {time:g} s: {q(time):.2f} W/m²; transferred {Q:.2f} J/m²")
assert np.isclose(el*(TL-contact),er*(contact-TR))

## Inspect both half-spaces and the early flux
The semi-infinite approximation requires the penetration distances to be small compared with specimen length. The flux diverges as time approaches zero, yet its time integral is finite. Change the right material's heat capacity at fixed conductivity.

In [ ]:
fig,(profile,flux)=plt.subplots(1,2,figsize=(10,4),layout="constrained")
width=5*np.sqrt(max(al,ar)*time)
x=np.linspace(-width,width,501)
for t,c in [(time/4,"#b94626"),(time,"#145c71")]:
    profile.plot(1000*x,temperature(x,t),color=c,label=f"{t:g} s")
profile.axvline(0,color="grey",ls=":")
profile.set(xlabel="Position (mm)",ylabel="Temperature (°C)",title="Continuity at the contact")
profile.legend()
tt=np.geomspace(.01*time,3*time,150)
flux.plot(tt,q(tt),color="#145c71")
flux.set_xscale("log")
if TL != TR: flux.set_yscale("symlog",linthresh=np.min(abs(q(tt)))/4)
flux.set(xlabel="Time (s)",ylabel="Left-to-right flux (W/m²)",title="Initially singular but integrable flux")
for ax in (profile,flux):ax.grid(alpha=.2)
plt.show()

## Independently integrate the energy in each body
Temperature deficits and excesses integrated normal to the contact, and multiplied by volumetric heat capacity, are energies per unit area. Both should agree with the integral of the interface flux.

In [ ]:
l=np.linspace(-12*np.sqrt(al*time),0,3001)
r=np.linspace(0,12*np.sqrt(ar*time),3001)
def trapezoid(values,grid):
    return np.sum((values[:-1]+values[1:])*np.diff(grid)/2)
lost=capacity_left*trapezoid(TL-temperature(l,time),l)
gained=capacity_right*trapezoid(temperature(r,time)-TR,r)
assert np.isclose(lost,Q,rtol=1e-6) and np.isclose(gained,Q,rtol=1e-6)
print("Spatial energy integrals:",lost,gained,"J/m²")